# TP2 - Sistemas Inteligentes: Recomendador grupal de películas y libros

**Caso de negocio:** un grupo de 5 amigos nunca se pone de acuerdo en qué ver o leer. Este sistema cruza los 5 perfiles de gustos, busca candidatos en una base de sinopsis (RAG) y devuelve UNA recomendación final justificada, en tono canchero argentino.

**Arquitectura:** `ingestion/` (TMDB + Google Books) -> `data/dataset.csv` -> `rag/` (embeddings + ChromaDB) -> `agent/` (cruza perfiles, RAG, rankea) -> `prompts/` (personalidad) -> respuesta final. Ver `docs/DISENO.md` en el repo para el detalle de cada decisión.

**Cómo correr esta notebook:**
1. Subir/clonar el repo completo a Colab (celda siguiente).
2. Cargar los secrets (`ANTHROPIC_API_KEY`, `TMDB_API_KEY`) en el panel de secrets de Colab (ícono de llave a la izquierda) con esos mismos nombres.
3. Correr las celdas en orden.

## 1. Setup del entorno

In [ ]:
# Si estás en Colab, clonar el repo (reemplazar por la URL real del repo del equipo)
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    !git clone https://github.com/<usuario-o-org>/<nombre-repo>.git repo
    %cd repo
    !pip install -q -r requirements.txt

In [ ]:
# Cargar API keys: en Colab desde userdata (Secrets), en local desde .env
import os

if IN_COLAB:
    from google.colab import userdata

    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")
    os.environ.setdefault("ANTHROPIC_MODEL", "claude-sonnet-5")
    os.environ.setdefault("CHROMA_PERSIST_DIR", "./data/chroma")
    os.environ.setdefault("DATASET_CSV_PATH", "./data/dataset.csv")
else:
    from dotenv import load_dotenv

    load_dotenv()

## 2. Ingesta de datos (`/ingestion`)

**Decisión de diseño:** si `data/dataset.csv` ya está commiteado en el repo, no hace
falta correr la ingesta de nuevo (ahorra tiempo y evita gastar la cuota de las APIs
el día de la defensa). Solo correr la celda de abajo si querés refrescar el dataset.

In [ ]:
REGENERAR_DATASET = False  # poner True para volver a pegarle a TMDB/Google Books

if REGENERAR_DATASET:
    !python -m ingestion.build_dataset

import pandas as pd

df = pd.read_csv(os.environ["DATASET_CSV_PATH"])
print(f"Dataset: {len(df)} filas")
df.head()

## 3. RAG: embeddings + ChromaDB (`/rag`)

**Decisión de diseño:** embeddings locales multilingües de HuggingFace en vez de
OpenAI, para no sumar una dependencia/costo extra al stack de Anthropic. Ver
`docs/DISENO.md` para el trade-off completo.

In [ ]:
from rag.vectorstore import construir_vectorstore

vectorstore = construir_vectorstore()
print(f"Vectorstore listo con {vectorstore._collection.count()} documentos.")

In [ ]:
# Prueba rápida del retriever
from rag.retriever import retriever

retriever("thriller psicológico con giros de guión", k=3)

## 4. Perfiles del grupo (`/perfiles`)

Cada integrante completó su `perfiles/<nombre>.json` (ver `perfiles/README.md` para el
esquema). Se cargan todos automáticamente.

In [ ]:
from agent.perfil import cargar_todos_los_perfiles

perfiles = cargar_todos_los_perfiles()
for p in perfiles:
    print(p.nombre)

## 5. Agente mediador (`/agent` + `/prompts`)

Cruza los 5 perfiles, busca candidatos vía RAG, los rankea por afinidad grupal y le
pide a Claude la recomendación final con el tono de `prompts/system_prompt.py`.

In [ ]:
from agent.mediador import recomendar_grupal

resultado = recomendar_grupal(perfiles)
print("Criterio de búsqueda usado:", resultado["criterio_busqueda"])
print()
print("=== Recomendación final ===")
print(resultado["recomendacion"])

In [ ]:
# Candidatos que consideró el agente, para mostrar en la defensa cómo llegó a la respuesta
import pandas as pd

pd.DataFrame(resultado["candidatos_rankeados"])

## 6. Casos de prueba para la defensa

Documentar acá 2-3 corridas con distintos perfiles/grupos para mostrar que el sistema
funciona con casos variados (pedido explícito de la rúbrica: "Funcionamiento del
sistema" y "Presentación y defensa").

In [ ]:
# TODO equipo: agregar acá 2-3 corridas de ejemplo con perfiles distintos y comentar
# el resultado (qué esperaban, qué dio, si tiene sentido).